# LezzetPOS hijyen modeli eğitimi (bone / önlük)

Bu defter yalnızca **eğitimi hızlandırmak** için Colab'ın ücretsiz ekran kartını kullanır. Çıkan `best.pt` dosyası indirilip
`tools/hygiene-camera/` klasörüne konur; tespit tamamen restoranın bilgisayarında, internetsiz çalışır.

**Adımlar:**
1. Üst menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç.
2. `hijyen-dataset.zip` dosyasını Google Drive'ının ana klasörüne (Drive'ım) yükle.
3. **Çalışma zamanı → Tümünü çalıştır**. Drive izni istendiğinde onayla.
4. Bitince son hücre `hijyen-model.zip` dosyasını indirir (Drive'a da kopyalar).

**Bağlantı koparsa:** Ara kayıtlar Drive'daki `hijyen-runs` klasörüne yazılır. Yeniden bağlanıp "Tümünü çalıştır" demen yeterli; eğitim kaldığı turdan devam eder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/hijyen-dataset.zip -d /content/hijyen
# data.yaml içindeki yolu Colab'daki klasöre göre düzelt
import re, pathlib
p = pathlib.Path('/content/hijyen/dataset/data.yaml')
p.write_text(re.sub(r'^path: .*$', 'path: /content/hijyen/dataset', p.read_text(encoding='utf-8'), flags=re.M), encoding='utf-8')
print(p.read_text(encoding='utf-8'))

In [ ]:
import os
from ultralytics import YOLO
# Ara kayıtlar Drive'a yazılır: Colab bağlantısı koparsa defteri yeniden çalıştırınca kaldığı turdan devam eder.
RUNS = '/content/drive/MyDrive/hijyen-runs'
last = f'{RUNS}/hijyen/weights/last.pt'
if os.path.exists(last):
    print('Önceki eğitim bulundu, kaldığı yerden devam ediliyor:', last)
    YOLO(last).train(resume=True)
else:
    # yolov8s: n'ye göre daha isabetli, restoran bilgisayarında hâlâ gerçek zamanlı çalışabilir.
    YOLO('yolov8s.pt').train(data='/content/hijyen/dataset/data.yaml', epochs=60, patience=15, imgsz=640, batch=32,
                             project=RUNS, name='hijyen', exist_ok=True, seed=42)

In [ ]:
# Hiç görülmemiş test bölümünde ölçüm (TÜBİTAK için sınıf bazında mAP)
best = YOLO('/content/drive/MyDrive/hijyen-runs/hijyen/weights/best.pt')
metrics = best.val(data='/content/hijyen/dataset/data.yaml', split='test', project='/content/drive/MyDrive/hijyen-runs', name='hijyen-test', exist_ok=True)
print('test mAP@0.5:', round(metrics.box.map50, 3), ' mAP@0.5:0.95:', round(metrics.box.map, 3))
for i, name in best.names.items():
    print(f'  {name:9s} mAP@0.5 = {metrics.box.ap50[i]:.3f}')

In [ ]:
# Model + eğitim grafikleri + test sonuçları tek dosyada (Drive'da da hijyen-runs klasöründe durur)
!cd /content/drive/MyDrive/hijyen-runs && zip -q -r /content/hijyen-model.zip hijyen hijyen-test
!cp /content/hijyen-model.zip /content/drive/MyDrive/
from google.colab import files
files.download('/content/hijyen-model.zip')